# 04 - Model Training
5 models on identical footing:
- **Logistic Regression**: linear baseline, fixed params, SimpleImputer + StandardScaler
- **DT / RF / XGBoost / LightGBM**: Optuna TPE, 100 trials each, 5-fold stratified CV on training set only. Test set never touched during HPO.
- Calibration: isotonic regression applied to winning model scores.

**Expected runtime:** LR ~2 min | DT ~10 min | RF ~30 min | XGB ~20 min | LGB ~20 min | Total ~80 min

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (average_precision_score, roc_auc_score,
    f1_score, precision_score, recall_score, brier_score_loss,
    precision_recall_curve, roc_curve)
import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from src.config import (PROCESSED_FILES, MODELS_DIR, OUTPUTS_DIR, CALIBRATION_SPLIT,
    FIGURES_DIR, TABLES_DIR,
    FEATURE_COLS, RANDOM_STATE, OPTUNA_N_TRIALS)
print(f"Imports OK | Optuna trials: {OPTUNA_N_TRIALS} per tree model | Features: {len(FEATURE_COLS)}")

Imports OK | Optuna trials: 100 per tree model | Features: 84


In [2]:
train_df = pd.read_csv(PROCESSED_FILES["train_set"], low_memory=False)
test_df  = pd.read_csv(PROCESSED_FILES["test_set"],  low_memory=False)
_meta_df = pd.read_csv(MODELS_DIR / "split_meta.csv")
meta = dict(zip(_meta_df["key"], _meta_df["value"]))
for _k in ["n_train","n_test","n_prospective","n_train_positive","n_train_negative"]:
    if _k in meta: meta[_k] = int(float(meta[_k]))
with open(MODELS_DIR / "feature_cols.txt") as _f:
    meta["feature_cols"] = [l.strip() for l in _f if l.strip()]
X_train = train_df[FEATURE_COLS].values
y_train = train_df["label"].values.astype(int)
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df["label"].values.astype(int)
SPW = round((y_train == 0).sum() / max((y_train == 1).sum(), 1), 4)
print(f"Train: {X_train.shape}  positive={y_train.mean():.2%}")
print(f"Test : {X_test.shape}   positive={y_test.mean():.2%}")
print(f"scale_pos_weight: {SPW}")
# sklearn DecisionTree and RandomForest crash on NaN (unless sklearn >= 1.4 with native NaN support)
# LightGBM and XGBoost handle NaN natively -- X_train used as-is for those
# LogisticRegression has its own Pipeline with SimpleImputer
from sklearn.impute import SimpleImputer as _TreeImp
_nan_count = np.isnan(X_train).sum()
print(f"NaN values in X_train: {_nan_count:,} -- creating median-imputed arrays for DT + RF")
_tree_imp  = _TreeImp(strategy="median")
X_train_i  = _tree_imp.fit_transform(X_train)   # imputed -- use for DT and RF
X_test_i   = _tree_imp.transform(X_test)        # imputed -- use for DT and RF

Train: (98926, 84)  positive=6.69%
Test : (94421, 84)   positive=4.07%
scale_pos_weight: 13.9367
NaN values in X_train: 542,226 -- creating median-imputed arrays for DT + RF


In [3]:
def bootstrap_ci(y_true, y_score, fn, n=500, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    s = []
    N = len(y_true)
    for _ in range(n):
        idx = rng.integers(0, N, N)
        yt, ys = y_true[idx], y_score[idx]
        if yt.sum() in (0, N): continue
        s.append(fn(yt, ys))
    return round(np.percentile(s, (1-ci)/2*100), 4), round(np.percentile(s, (1+ci)/2*100), 4)

def ks_stat(y_true, y_score):
    from scipy import stats
    return round(stats.ks_2samp(y_score[y_true==1], y_score[y_true==0]).statistic, 4)

def get_metrics(name, y_true, y_score):
    best_f1, best_thr = 0, 0.5
    for thr in np.arange(0.1, 0.9, 0.01):
        yp = (y_score >= thr).astype(int)
        if yp.sum() == 0: continue
        f = f1_score(y_true, yp, zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    yp_best = (y_score >= best_thr).astype(int)
    ap  = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    ap_lo, ap_hi   = bootstrap_ci(y_true, y_score, average_precision_score)
    auc_lo, auc_hi = bootstrap_ci(y_true, y_score, roc_auc_score)
    return {
        "model": name,
        "avg_precision": round(ap,4), "ap_ci_lo": ap_lo, "ap_ci_hi": ap_hi,
        "auc_roc": round(auc,4), "auc_ci_lo": auc_lo, "auc_ci_hi": auc_hi,
        "f1": round(best_f1,4),
        "precision": round(precision_score(y_true, yp_best, zero_division=0),4),
        "recall":    round(recall_score(y_true, yp_best, zero_division=0),4),
        "ks_stat":    ks_stat(y_true, y_score),
        "brier_score": round(brier_score_loss(y_true, y_score),4),
        "threshold":   round(best_thr, 2),
    }
print("Metrics helpers ready (bootstrap 500 resamples, 95% CI)")

Metrics helpers ready (bootstrap 500 resamples, 95% CI)


In [4]:
# Logistic Regression (linear baseline)
print("Training Logistic Regression (linear baseline, fixed params)...")
lr_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("clf",     LogisticRegression(class_weight="balanced", max_iter=1000,
                                   random_state=RANDOM_STATE)),
])
lr_pipe.fit(X_train, y_train)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]
print("  DONE")

Training Logistic Regression (linear baseline, fixed params)...
  DONE


In [5]:
# Decision Tree (Optuna)
print(f"Optuna ({OPTUNA_N_TRIALS} trials) - Decision Tree...")
def dt_obj(trial):
    p = {"max_depth": trial.suggest_int("max_depth", 3, 20),
         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 200),
         "min_samples_split": trial.suggest_int("min_samples_split", 2, 100),
         "class_weight": "balanced", "random_state": RANDOM_STATE}
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    s  = [average_precision_score(y_train[v], DecisionTreeClassifier(**p).fit(X_train_i[t], y_train[t]).predict_proba(X_train_i[v])[:,1])
          for t, v in cv.split(X_train_i, y_train)]
    return np.mean(s)
dt_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
dt_study.optimize(dt_obj, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
dt_best = {**dt_study.best_params, "class_weight":"balanced", "random_state":RANDOM_STATE}
dt = DecisionTreeClassifier(**dt_best).fit(X_train_i, y_train)
dt_proba = dt.predict_proba(X_test_i)[:, 1]
print(f"  DONE  best CV AP={dt_study.best_value:.4f}")

Optuna (100 trials) - Decision Tree...


  0%|          | 0/100 [00:00<?, ?it/s]

  DONE  best CV AP=0.5122


In [6]:
# Random Forest (Optuna)
print(f"Optuna ({OPTUNA_N_TRIALS} trials) - Random Forest (~30 min)...")
def rf_obj(trial):
    p = {"n_estimators": trial.suggest_int("n_estimators", 100, 400),
         "max_depth": trial.suggest_int("max_depth", 5, 25),
         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 100),
         "max_features": trial.suggest_float("max_features", 0.3, 1.0),
         "class_weight": "balanced_subsample", "n_jobs": -1, "random_state": RANDOM_STATE}
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    from sklearn.ensemble import RandomForestClassifier as RFC
    s  = [average_precision_score(y_train[v], RFC(**p).fit(X_train_i[t], y_train[t]).predict_proba(X_train_i[v])[:,1])
          for t, v in cv.split(X_train_i, y_train)]
    return np.mean(s)
rf_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
rf_study.optimize(rf_obj, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
rf_best = {**rf_study.best_params, "class_weight":"balanced_subsample", "n_jobs":-1, "random_state":RANDOM_STATE}
rf = RandomForestClassifier(**rf_best).fit(X_train_i, y_train)
rf_proba = rf.predict_proba(X_test_i)[:, 1]
print(f"  DONE  best CV AP={rf_study.best_value:.4f}")

Optuna (100 trials) - Random Forest (~30 min)...


  0%|          | 0/100 [00:00<?, ?it/s]

  DONE  best CV AP=0.6820


In [7]:
# XGBoost (Optuna)
print(f"Optuna ({OPTUNA_N_TRIALS} trials) - XGBoost (CV only, no test set)...")
def xgb_obj(trial):
    p = {"n_estimators": trial.suggest_int("n_estimators", 200, 1000),
         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
         "max_depth": trial.suggest_int("max_depth", 3, 9),
         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
         "scale_pos_weight": SPW, "eval_metric": "aucpr",
         "random_state": RANDOM_STATE, "verbosity": 0, "n_jobs": -1}
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    s  = [average_precision_score(y_train[v], xgb.XGBClassifier(**p).fit(X_train[t], y_train[t], verbose=False).predict_proba(X_train[v])[:,1])
          for t, v in cv.split(X_train, y_train)]
    return np.mean(s)
xgb_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
xgb_study.optimize(xgb_obj, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
xgb_best = {**xgb_study.best_params, "scale_pos_weight":SPW, "eval_metric":"aucpr",
             "random_state":RANDOM_STATE, "verbosity":0, "n_jobs":-1}
xgb_model = xgb.XGBClassifier(**xgb_best).fit(X_train, y_train, verbose=False)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
print(f"  DONE  best CV AP={xgb_study.best_value:.4f}")
import joblib as _jl
_jl.dump(xgb_model, MODELS_DIR / "xgboost_model.joblib", compress=3)
print("  XGBoost saved as xgboost_model.joblib")

Optuna (100 trials) - XGBoost (CV only, no test set)...


  0%|          | 0/100 [00:00<?, ?it/s]

  DONE  best CV AP=0.7794
  XGBoost saved as xgboost_model.joblib


In [8]:
# LightGBM (Optuna)
print(f"Optuna ({OPTUNA_N_TRIALS} trials) - LightGBM...")
def lgb_obj(trial):
    p = {"n_estimators": trial.suggest_int("n_estimators", 200, 1000),
         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
         "max_depth": trial.suggest_int("max_depth", 3, 9),
         "num_leaves": trial.suggest_int("num_leaves", 20, 150),
         "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
         "scale_pos_weight": SPW, "random_state": RANDOM_STATE,
         "verbosity": -1, "n_jobs": -1}
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    s  = []
    for t, v in cv.split(X_train, y_train):
        m = lgb.LGBMClassifier(**p)
        m.fit(X_train[t], y_train[t],
              eval_set=[(X_train[v], y_train[v])],
              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        s.append(average_precision_score(y_train[v], m.predict_proba(X_train[v])[:,1]))
    return np.mean(s)
lgb_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
lgb_study.optimize(lgb_obj, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
lgb_best = {**lgb_study.best_params, "scale_pos_weight":SPW,
            "random_state":RANDOM_STATE, "verbosity":-1, "n_jobs":-1}
lgbm_model = lgb.LGBMClassifier(**lgb_best)
lgbm_model.fit(X_train, y_train)
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
print(f"  DONE  best CV AP={lgb_study.best_value:.4f}")

Optuna (100 trials) - LightGBM...


  0%|          | 0/100 [00:00<?, ?it/s]

  DONE  best CV AP=0.5974


In [9]:
# Model comparison
rows = [
    get_metrics("Logistic Regression", y_test, lr_proba),
    get_metrics("Decision Tree",       y_test, dt_proba),
    get_metrics("Random Forest",       y_test, rf_proba),
    get_metrics("XGBoost",             y_test, xgb_proba),
    get_metrics("LightGBM",            y_test, lgbm_proba),
]
comp_df = pd.DataFrame(rows).sort_values("avg_precision", ascending=False).reset_index(drop=True)
comp_df.to_csv(TABLES_DIR / "model_comparison.csv", index=False)
print("MODEL COMPARISON (sorted by Average Precision)")
print("All tree models: equal Optuna budget, 5-fold CV, test set never used during HPO")
print("="*92)
print(f"  {'Model':<22} {'AP':>7} {'AP 95%CI':>14} {'AUC':>7} {'AUC 95%CI':>14} {'F1':>7} {'KS':>7} {'Brier':>7}")
print("-"*92)
for i, r in comp_df.iterrows():
    star = " *" if i == 0 else ""
    print(f"  {r['model']:<22} {r['avg_precision']:>7.4f} [{r['ap_ci_lo']:.3f},{r['ap_ci_hi']:.3f}] "
          f"{r['auc_roc']:>7.4f} [{r['auc_ci_lo']:.3f},{r['auc_ci_hi']:.3f}] "
          f"{r['f1']:>7.4f} {r['ks_stat']:>7.4f} {r['brier_score']:>7.4f}{star}")
print("\n* = winner (highest Average Precision)")

MODEL COMPARISON (sorted by Average Precision)
All tree models: equal Optuna budget, 5-fold CV, test set never used during HPO
  Model                       AP       AP 95%CI     AUC      AUC 95%CI      F1      KS   Brier
--------------------------------------------------------------------------------------------
  XGBoost                 0.6298 [0.615,0.646]  0.9412 [0.937,0.945]  0.5933  0.7400  0.0272 *
  LightGBM                0.6012 [0.586,0.619]  0.9313 [0.927,0.935]  0.5752  0.7148  0.0287
  Logistic Regression     0.5353 [0.521,0.554]  0.9679 [0.965,0.971]  0.6299  0.8463  0.0532
  Random Forest           0.4522 [0.435,0.469]  0.8810 [0.875,0.887]  0.4489  0.6074  0.0725
  Decision Tree           0.1771 [0.166,0.189]  0.7846 [0.777,0.793]  0.2468  0.4562  0.2297

* = winner (highest Average Precision)


In [10]:
# Save all models + calibration
import joblib as _jl
winner_name = comp_df.iloc[0]["model"]
winner_row  = comp_df.iloc[0]
model_map = {
    "Logistic Regression": lr_pipe,
    "Decision Tree":        dt,
    "Random Forest":        rf,
    "XGBoost":              xgb_model,
    "LightGBM":             lgbm_model,
}
winner_model = model_map[winner_name]
print("Saving all models as .joblib:")
for mname, mobj in model_map.items():
    fname = mname.lower().replace(" ","_") + "_model.joblib"
    _jl.dump(mobj, MODELS_DIR / fname, compress=3)
    tag = " <- WINNER" if mname == winner_name else ""
    print(f"  {fname}{tag}")

# Isotonic calibration
winner_proba_raw = dict(zip([m for m,_ in model_map.items()],[p for p in [lr_proba,dt_proba,rf_proba,xgb_proba,lgbm_proba]]))[winner_name]
cal_idx  = np.arange(0, len(y_test), 2)
eval_idx = np.arange(1, len(y_test), 2)
iso_reg  = IsotonicRegression(out_of_bounds="clip")
iso_reg.fit(winner_proba_raw[cal_idx], y_test[cal_idx])
winner_proba_cal = iso_reg.predict(winner_proba_raw)
_jl.dump(iso_reg, MODELS_DIR / "isotonic_calibrator.joblib", compress=3)
print(f"\nIsotonic calibrator saved ({len(iso_reg.X_thresholds_)} threshold pairs)")
print(f"Raw score range      : [{winner_proba_raw.min():.4f}, {winner_proba_raw.max():.4f}]")
print(f"Calibrated range     : [{winner_proba_cal.min():.4f}, {winner_proba_cal.max():.4f}]")

# Save winner meta
_wmeta = [
    {"key": "winner_name",    "value": winner_name},
    {"key": "winner_thr",     "value": float(winner_row["threshold"])},
    {"key": "avg_precision",  "value": float(winner_row["avg_precision"])},
    {"key": "auc_roc",        "value": float(winner_row["auc_roc"])},
    {"key": "f1",             "value": float(winner_row["f1"])},
    {"key": "ks_stat",        "value": float(winner_row["ks_stat"])},
    {"key": "brier_score",    "value": float(winner_row["brier_score"])},
    {"key": "optuna_n_trials","value": OPTUNA_N_TRIALS},
    {"key": "optuna_cv_folds","value": 5},
    {"key": "scale_pos_weight","value": SPW},
]
pd.DataFrame(_wmeta).to_csv(MODELS_DIR / "winner_meta.csv", index=False)
for _pname, _params in [("dt", dt_study.best_params), ("rf", rf_study.best_params),
                          ("xgb", xgb_study.best_params), ("lgb", lgb_study.best_params)]:
    pd.DataFrame([{"param": k, "value": v} for k,v in _params.items()]).to_csv(
        MODELS_DIR / f"best_params_{_pname}.csv", index=False)
with open(MODELS_DIR / "feature_cols.txt", "w") as _f:
    _f.write("\n".join(FEATURE_COLS))
print(f"\nWINNER: {winner_name}")
print(f"  AP  : {winner_row['avg_precision']:.4f}  [{winner_row['ap_ci_lo']:.3f}, {winner_row['ap_ci_hi']:.3f}]")
print(f"  AUC : {winner_row['auc_roc']:.4f}  [{winner_row['auc_ci_lo']:.3f}, {winner_row['auc_ci_hi']:.3f}]")
print(f"  F1  : {winner_row['f1']:.4f}")
print(f"  KS  : {winner_row['ks_stat']:.4f}")
print(f"  Brier: {winner_row['brier_score']:.4f}")
print()
print("BENCHMARK CHECK")
print("="*55)
print("  Du Jardin (2021) AUC benchmark : 0.83 - 0.88  (ratio-based)")
print("  Proposal RQ1 AUC target        : >= 0.75  (metadata-only, clean split)")
print(f"  Winner AUC : {winner_row['auc_roc']:.4f}  {'OK TARGET MET' if winner_row['auc_roc']>=0.75 else 'BELOW TARGET'}")
print()
print("Ready for 05_anomaly_detection.ipynb")

Saving all models as .joblib:
  logistic_regression_model.joblib
  decision_tree_model.joblib
  random_forest_model.joblib
  xgboost_model.joblib <- WINNER
  lightgbm_model.joblib

Isotonic calibrator saved (120 threshold pairs)
Raw score range      : [0.0000, 1.0000]
Calibrated range     : [0.0000, 1.0000]

WINNER: XGBoost
  AP  : 0.6298  [0.615, 0.646]
  AUC : 0.9412  [0.937, 0.945]
  F1  : 0.5933
  KS  : 0.7400
  Brier: 0.0272

BENCHMARK CHECK
  Du Jardin (2021) AUC benchmark : 0.83 - 0.88  (ratio-based)
  Proposal RQ1 AUC target        : >= 0.75  (metadata-only, clean split)
  Winner AUC : 0.9412  OK TARGET MET

Ready for 05_anomaly_detection.ipynb


## Standardisation note

The tree-based models (XGBoost, LightGBM, Random Forest, Decision Tree) and the
logistic-regression baseline in its pipeline do not require feature standardisation.
Gradient-boosted and bagged trees split on feature rank rather than magnitude, so
they are invariant to monotonic rescaling; the logistic-regression baseline is fit
inside a pipeline that scales internally. Standardisation is required only for the
scale-sensitive Stage 2 models, where Euclidean distance (Isolation Forest) and the
hazard function (Cox) are affected by feature magnitude. A `StandardScaler` is
therefore fitted on the training features only and saved for Stage 2 to apply to
test and prospective data, keeping the scaling free of test-set information.

In [11]:
# Fit a train-only StandardScaler for the scale-sensitive Stage 2 models
# (Isolation Forest, Cox). Tree models above use the raw features unchanged.
from sklearn.preprocessing import StandardScaler
import joblib as _jl2
_scaler = StandardScaler().fit(train_df[FEATURE_COLS].fillna(0).values)
_jl2.dump(_scaler, MODELS_DIR / "stage2_scaler.joblib", compress=3)
print("Saved train-only StandardScaler for Stage 2:", MODELS_DIR / "stage2_scaler.joblib")


Saved train-only StandardScaler for Stage 2: E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17\models\stage2_scaler.joblib
